# Graph Analysis Ideas for MCP-Bench LLM Graph Outputs

> Analysis framework for comparing `ground_truth_graph` vs `llm_graph` across tasks, models, repetitions, and fuzzy variants.

---

## 1. Graph Structure & Topology

### Node-level Analysis
- Node degree distribution (in/out degree for directed graphs)
- Betweenness, closeness, and eigenvector centrality — which tools are "hubs"?
- Which nodes appear in LLM graphs but not ground truth (**hallucinated tools**) and vice versa (**missed tools**)
- Node precision / recall / F1 across tasks and models
- Which specific tools are most frequently hallucinated vs. most frequently missed
- Tool **"difficulty" score**: how often each tool is correctly placed across all runs

### Edge-level Analysis
- Edge precision / recall / F1 — are the right dependencies captured?
- Which dependency edges are most commonly missed or hallucinated
- **Directionality errors**: correct tools connected but in the wrong order
- **Transitive closure comparison** — does the LLM capture implied dependencies even if not direct ones?

### Global Graph Metrics
- Graph edit distance (GED) between LLM graph and ground truth
- Jaccard similarity on node sets and edge sets separately
- Graph density comparison — do LLM graphs tend to be sparser or denser?
- Number of connected components — does the LLM fragment workflows?
- Longest path length comparison — does the LLM understand deep chains?
- Diameter comparison
- Clustering coefficient — does the LLM over- or under-cluster tools?

### Structural Pattern Analysis
- How well does the LLM preserve **parallel branches** vs. **sequential chains**?
- Identification of subgraph isomorphisms — which substructures are reliably reproduced?
- **DAG vs. cycle detection** — does the LLM ever introduce cycles into what should be a DAG?
- **Critical path preservation**: is the longest dependency chain captured correctly?

---

## 2. Repetition Analysis *(Same prompt, repeated runs)*

- Graph similarity variance across repetitions for the same `task_id` + `model` — **consistency score**
- Node set stability: which nodes appear in all N repetitions vs. only some?
- Edge set stability: which edges are "certain" vs. "flickering" across runs?
- Coefficient of variation (CV) of graph size (nodes/edges) across repetitions
- Do hallucinated nodes tend to be **consistent hallucinations** or random noise?
- Entropy of tool selection across repetitions — high entropy = unstable planning
- Pairwise similarity matrix across all repetitions → cluster to find stable vs. chaotic tasks
- **Warm-up effect?** Does repetition index (0, 1, 2, ...) correlate with quality?
- **Bimodal distribution detection** — do some tasks have two "modes" the model switches between?

---

## 3. Fuzzy Variant Analysis *(Same ground truth, different fuzzy descriptions)*

- Does the phrasing of the fuzzy description affect which tools are selected?
- Correlation between fuzzy description **length / complexity** and graph quality
- Which ground truth graphs are **"easy"** (consistently reproduced across fuzzings) vs. **"hard"**?
- Does the presence of domain-specific vocabulary in the fuzzy prompt help tool retrieval?
- **Sensitivity analysis**: small wording changes → large graph changes (butterfly effect)?
- Do different fuzzy variants of the same task produce systematically different **error types**?
- Ground truth graph complexity vs. LLM accuracy — do harder ground truths see steeper degradation?

---

## 4. Model Comparison

- Per-model node F1 and edge F1 across all tasks
- Which models **hallucinate more** vs. which models **miss more** (precision vs. recall tradeoff)
- Model **consistency**: which model produces the most stable graphs across repetitions?
- **Model ranking stability**: does task difficulty ranking stay consistent across models?
- Do larger/stronger models produce structurally closer graphs, or just larger ones?
- Which models handle **parallel branches** better than **sequential chains**, or vice versa?
- **Cross-model agreement**: when do all models agree on the same wrong graph?
- **Model-specific systematic biases** — e.g., does one model always prefer certain tools?

---

## 5. Task Difficulty & Clustering

- Define a **"task difficulty" score** and correlate with: ground truth size, domain, number of servers
- Cluster tasks by ground truth graph structure — which structural types are hardest?
- **Single-server vs. multi-server** tasks: how much does cross-server orchestration hurt accuracy?
- Task domain vs. performance — are finance tasks harder than geography tasks?
- Does ground truth graph **depth** (longest path) predict LLM performance better than graph size?
- Tasks where **all models fail** completely — what do they have in common?
- Tasks where **all models succeed** — trivially easy or genuinely well-structured?

---

## 6. Error Taxonomy

- Classify errors into types:
  - Missing node
  - Extra node (hallucination)
  - Missing edge
  - Extra edge
  - Wrong edge direction
  - Wrong parameterisation
- Are certain error types **correlated** — e.g., if a node is missed, are its downstream edges always also missing?
- **Error propagation**: if an early node in the chain is wrong, how far downstream does the error cascade?
- Which tools are **"safe anchors"** — almost always correctly included regardless of model or phrasing?
- Which tools are **"traps"** — frequently hallucinated when they shouldn't be?
- Do errors cluster at the **beginning, middle, or end** of dependency chains?

---

## 7. Information-Theoretic & Statistical

- Mutual information between ground truth graph structure and LLM graph structure
- How much of the variance in LLM graph quality is explained by: **model**, **task**, **repetition**, **fuzzy variant**?
- ANOVA / mixed-effects model: decompose variance sources
- Bootstrap confidence intervals on all similarity metrics
- Correlation between graph similarity metrics — do GED, Jaccard, and F1 tell the same story or different stories?
- Statistical significance testing when comparing models on the same task set

---

## 8. Connecting to MCP-Bench Evaluation Scores

- Correlation between graph similarity metrics and the **LLM-as-a-Judge scores** (task fulfillment, grounding, dependency awareness, parallelism)
- Does **dependency awareness score** correlate with edge F1?
- Does **parallelism efficiency score** correlate with parallel branch preservation?
- Can graph structure metrics **predict overall score** without running the full judge pipeline?
- Which single graph metric is the **best predictor** of downstream task performance?

---

## 9. Visualisations

| Visualisation | What it shows |
|---|---|
| Heatmap of pairwise task similarity | Ground truth graph structure clustering |
| Tool confusion matrix | Which tools are swapped for which |
| Sankey diagram of tool flow | How often tool A → tool B in LLM vs. ground truth |
| Radar / spider charts | Per-model comparison across multiple graph metrics |
| Graph overlays | Ground truth + LLM graph with matched / missing / hallucinated colour coding |
| Edge stability timeline | Which edges solidify or dissolve across repetitions |
| Dendrogram | Task clustering by difficulty profile |

---

## 10. Meta & Methodological

- Does the number of **distractor tools** correlate with graph degradation?
- **Vocabulary alignment effect** — tasks where tool names match fuzzy description words more closely → better graphs?
- Does the **order** in which tools are listed in the prompt affect which ones appear in LLM graphs?
- Length of ground truth task description vs. LLM graph accuracy
- Can a simple classifier predict **"will this task fail?"** from ground truth graph features alone?

---

*Generated as an analysis planning document for MCP-Bench LLM graph output exploration.*

### Similarity approach ###

Look at determining similarity scoring and generate a similarity matrix. use this to calculate:


mean_similarityOverall consistency — how alike runs are on averagestd_similarityStability — low std means the agent reliably picks the same planmin_similarityWorst-case divergence — the most different pair of plansn_clustersWhether the agent has distinct "modes" (e.g. a short path and a long path)outlier_indicesWhich specific runs deviated — useful for debuggingcentroid_idxThe most "representative" run to use as a reference plan